# 16 端侧扩散模型与图像生成

## 为什么单独成章？

LLM 是端侧主战场，但 **手机上的文生图 / 图生图**（SD-Turbo、LCM、SVD 等）同样重要：步数少、分辨率受限、UNet/DiT + VAE 的内存峰值与 LLM 完全不同。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}")

## 16.1 扩散采样步数与延迟关系

In [ ]:
def diffusion_latency_ms(steps, step_ms=45, vae_ms=30):
    return steps * step_ms + vae_ms


print("=== 步数 vs 延迟（示意）===")
for steps, name in [(50, "经典 SD"), (20, "加速采样"), (8, "LCM"), (4, "SD-Turbo/Lightning"), (1, "极端蒸馏")]:
    print(f"{name:16s} steps={steps:2d}  ≈{diffusion_latency_ms(steps):4.0f} ms")
print("端侧目标通常 ≤ 1s：优先 4–8 步蒸馏模型。")

## 16.2 玩具 UNet 一步更新 + INT8 权重

In [ ]:
class TinyResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1), nn.ReLU(),
            nn.Conv2d(c, c, 3, padding=1),
        )

    def forward(self, x):
        return x + self.net(x)


class TinyUNet(nn.Module):
    def __init__(self, c=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(3, c, 3, padding=1), TinyResBlock(c))
        self.mid = TinyResBlock(c)
        self.dec = nn.Conv2d(c, 3, 3, padding=1)

    def forward(self, x, t_embed):
        h = self.enc(x) + t_embed
        h = self.mid(h)
        return self.dec(h)


def ddim_step(x, eps, alpha_t, alpha_prev):
    """极简 DDIM 更新（教学）。"""
    x0 = (x - (1 - alpha_t).sqrt() * eps) / alpha_t.sqrt()
    return alpha_prev.sqrt() * x0 + (1 - alpha_prev).sqrt() * eps


unet = TinyUNet()
x = torch.randn(1, 3, 32, 32)
t_embed = torch.zeros_like(x[:, :16])
# 通道对齐：用均值广播
t_embed = torch.zeros(1, 16, 32, 32)
eps = unet(x, t_embed)
x_next = ddim_step(x, eps, alpha_t=torch.tensor(0.8), alpha_prev=torch.tensor(0.6))
n = sum(p.numel() for p in unet.parameters())
print(f"eps={tuple(eps.shape)} x'={tuple(x_next.shape)} params={n:,}")

## 16.3 端侧优化清单

1. **步数蒸馏**（LCM / Turbo）：50→4 步。
2. **权重 INT8/INT4 + 激活 FP16**：UNet 对量化相对鲁棒。
3. **VAE tiling / 小分辨率**：512→384/256，或分块解码。
4. **ControlNet / LoRA 按需加载**：常驻基座，插件热插拔。
5. **与 LLM 共存**：文生图时卸载 LLM 权重或权重流式，避免 OOM。

In [ ]:
def mobile_sd_memory_mb(unet_b=0.86, vae_b=0.08, text_b=0.12, bits=8, peak_act_mb=350):
    w = (unet_b + vae_b + text_b) * 1e9 * bits / 8 / 1e6
    return w + peak_act_mb


for bits in (16, 8, 4):
    print(f"W{bits}: 峰值约 {mobile_sd_memory_mb(bits=bits):.0f} MB（示意）")

## 小结

端侧扩散的核心不是“再砍一个 attention”，而是 **少步数 + 控分辨率 + 与 LLM 分时复用内存**。手机相册/壁纸生成场景优先选 Turbo/LCM 系模型。